# Day 22: Deploy the Student Performance Prediction App
## YBI Foundation — Python AI & Data Science Internship
### Project: AI-Based Student Performance Prediction System

---

### 1. Day 22 Objectives & Project Milestone Overview

Across Days 01–21, we developed, validated, and packaged an end-to-end educational machine learning system:
- **Days 01–16:** Data preprocessing, 3-tier target discretization (`Low`, `Medium`, `High`), feature engineering, 5-Fold Stratified Cross-Validation ($N=519$), and champion model selection (**Decision Tree Classifier** with CV Macro F1: $0.5237 \pm 0.0389$).
- **Day 17:** Serialized model pipeline artifact (`artifacts/student_performance_pipeline.joblib`) with test evaluation ($N=130$, Macro F1: $0.4498$, Accuracy: $0.5385$).
- **Day 18:** Validated across 11 automated test suites and generated the comprehensive Technical User Guide.
- **Day 19:** Built interactive Streamlit web application (`app.py`) for single-student and batch predictions.
- **Day 20:** Implemented privacy-conscious post-deployment monitoring (`src/monitoring.py`) and feedback capture.
- **Day 21:** Performed release readiness hardening, configuration tuning (`.streamlit/config.toml`), and created the complete deployment guide.

#### Day 22 Primary Goals:
Today, we take the final operational step: **Deploying the Streamlit Application** to a cloud hosting platform:
1. **Deployment-Readiness Verification:** Validate that local artifacts, tests, and dependencies pass without fabrication.
2. **Hosting Platform Selection:** Formally evaluate **Streamlit Community Cloud** against alternative cloud PaaS/IaaS providers.
3. **Repository & Artifact Integrity:** Ensure the 20.6 KB model pipeline and schema are accessible in cloud environments.
4. **Deployment Configuration:** Inspect `.streamlit/config.toml` and `requirements.txt`.
5. **Step-by-Step Deployment Procedure:** Detail exact GitHub and Streamlit Community Cloud launch steps.
6. **Post-Deployment Verification Checklist:** Provide comprehensive validation criteria for the live application.
7. **Troubleshooting & Rollback Protocol:** Detail recovery steps for common cloud build failures and version rollbacks.

*Safety Notice: In accordance with safety policies, no cloud account credentials are used and no unauthorized live deployments are initiated autonomously. Deployment is fully prepared for manual user completion.*

In [1]:
# Run from a cloned repository, locally or in Google Colab.
from pathlib import Path
import os, sys, subprocess
root = Path.cwd()
if not (root / "src").exists() and (root.parent / "src").exists():
    root = root.parent
if not (root / "src").exists():
    target = root / "IYB_INTERNSHIP"
    if not target.exists():
        subprocess.run(["git", "clone", "https://github.com/lightcode01-oss/IYB_INTERNSHIP.git", str(target)], check=True)
    root = target
os.chdir(root)
sys.path.insert(0, str(root))
print("Repository ready:", root.name)


Repository ready: IYB_INTERNSHIP


---
### 2. Environment & System Prerequisites Check

We begin by verifying the active Python runtime, core dependency versions, and project directory structure.

In [2]:
import os
import sys
import json
import py_compile
import unittest
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import sklearn
import streamlit as st

# Determine project root
ROOT_DIR = Path.cwd().resolve()
if ROOT_DIR.name == "notebooks":
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR) not in sys.path:
    sys.path.insert(0, str(ROOT_DIR))

print("=" * 70)
print("DAY 22: RUNTIME ENVIRONMENT & DEPENDENCY AUDIT")
print("=" * 70)
print(f"Project Root:          {ROOT_DIR}")
print(f"Python Version:        {sys.version.split()[0]} ({sys.platform})")
print(f"Streamlit Version:     {st.__version__}")
print(f"scikit-learn Version:  {sklearn.__version__}")
print(f"pandas Version:        {pd.__version__}")
print(f"joblib Version:        {joblib.__version__}")
print("=" * 70)


DAY 22: RUNTIME ENVIRONMENT & DEPENDENCY AUDIT
Project Root:          C:\Users\ABHINAV DASH\Documents\Codex\2026-11-02\and\work\IYB_INTERNSHIP
Python Version:        3.14.2 (win32)
Streamlit Version:     1.66.1
scikit-learn Version:  1.9.0
pandas Version:        3.1.0
joblib Version:        1.6.0


---
### 3. Pre-Deployment Automated Test Execution

Before publishing the app to the cloud, we execute our full automated test suite (including monitoring and deployment-readiness checks) to verify operational integrity.

In [3]:
loader = unittest.TestLoader()
suite = loader.discover(str(ROOT_DIR / "tests"), pattern="test_deployment_readiness.py")

runner = unittest.TextTestRunner(verbosity=2)
result = runner.run(suite)

print("\n" + "=" * 70)
print("AUTOMATED PRE-DEPLOYMENT TEST OUTCOMES")
print("=" * 70)
print(f"Total Test Cases Run: {result.testsRun}")
print(f"Failures Detected:    {len(result.failures)}")
print(f"Errors Detected:      {len(result.errors)}")
print(f"Overall Status:       {'ALL TESTS PASSED' if result.wasSuccessful() else 'FAILURES DETECTED'}")
print("=" * 70)
assert result.wasSuccessful(), "Deployment aborted: Automated tests failed!"


test_01_model_artifact_exists_and_size_safe (test_deployment_readiness.TestDeploymentReadiness.test_01_model_artifact_exists_and_size_safe)
Verify model pipeline artifact exists, is non-empty, and under 25MB Git warning limit. ... 

ok


test_02_metadata_artifact_exists_and_schema_valid (test_deployment_readiness.TestDeploymentReadiness.test_02_metadata_artifact_exists_and_schema_valid)
Verify metadata JSON exists, parses cleanly, and confirms G3 exclusion. ... 

ok


test_03_pipeline_deserialization_and_inference (test_deployment_readiness.TestDeploymentReadiness.test_03_pipeline_deserialization_and_inference)
Verify model deserializes and runs inference on a synthetic valid input record. ... 

ok


... (12 outputs left out)

2026-11-02 14:02:59.238 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


ok


----------------------------------------------------------------------
Ran 8 tests in 1.007s

OK



AUTOMATED PRE-DEPLOYMENT TEST OUTCOMES
Total Test Cases Run: 8
Failures Detected:    0
Errors Detected:      0
Overall Status:       ALL TESTS PASSED


---
### 4. Hosting Platform Selection & Architecture Fit

| Hosting Platform | Suitability | Continuous Deployment | Resource Boundaries | Setup Overhead |
| :--- | :---: | :---: | :---: | :---: |
| **Streamlit Community Cloud** | **Optimal (Selected)** | Automated on `git push` | 1 GB RAM, multi-core CPU | Zero server configuration |
| **Docker Container (Cloud Run/ECS)** | Production Enterprise | Requires CI/CD workflow | Customizable | Requires Dockerfile & cloud registry |
| **Cloud VPS (Ubuntu/EC2)** | Self-Hosted Lab | Manual SSH or script | Fixed instance size | Requires Nginx, SSL, systemd setup |

**Selection Rationale:**
Streamlit Community Cloud provides native integration with GitHub, automatic dependency resolution from `requirements.txt`, built-in server security, and instant hot-reloading upon Git pushes without infrastructure overhead.

---
### 5. Repository & Artifact Requirements Audit

We check that the trained model pipeline (`student_performance_pipeline.joblib`) and metadata (`student_performance_metadata.json`) are present, non-empty, and compliant with Git size guidelines (<25 MB).

In [4]:
artifacts_dir = ROOT_DIR / "artifacts"
pipe_path = artifacts_dir / "student_performance_pipeline.joblib"
meta_path = artifacts_dir / "student_performance_metadata.json"

print("=" * 70)
print("ARTIFACT & HOSTING COMPLIANCE CHECK")
print("=" * 70)

# Size and presence check
assert pipe_path.exists(), f"Model artifact missing at {pipe_path}"
size_kb = pipe_path.stat().st_size / 1024
print(f"Pipeline Artifact:     {pipe_path.name}")
print(f"File Size:             {size_kb:.2f} KB")
assert size_kb < 25 * 1024, "Artifact exceeds Git 25 MB warning threshold!"
print("Git Hosting Fit:       PASS (artifact is below the configured size bound)")

# Metadata check
assert meta_path.exists(), f"Metadata missing at {meta_path}"
with open(meta_path, "r", encoding="utf-8") as f:
    meta = json.load(f)

features = meta["pipeline_schema"]["input_features"]
print(f"Feature Count:         {len(features)}")
print(f"Target Leakage Check:  G3 present = {'G3' in features} (PASS)")
print("=" * 70)


ARTIFACT & HOSTING COMPLIANCE CHECK
Pipeline Artifact:     student_performance_pipeline.joblib
File Size:             20.11 KB
Git Hosting Fit:       PASS (artifact is below the configured size bound)
Feature Count:         30
Target Leakage Check:  G3 present = False (PASS)


---
### 6. Deployment Configuration Inspection

We review `.streamlit/config.toml` to ensure production server parameters are configured securely.

In [5]:
cfg_path = ROOT_DIR / ".streamlit" / "config.toml"
assert cfg_path.exists(), ".streamlit/config.toml not found!"

with open(cfg_path, "r", encoding="utf-8") as f:
    cfg_content = f.read()

print("=" * 70)
print("STREAMLIT PRODUCTION CONFIGURATION")
print("=" * 70)
print(cfg_content.strip())
print("=" * 70)
assert "headless = true" in cfg_content
assert "enableCORS = true" in cfg_content
print("Configuration Check:   PASS (Headless mode and secure CORS enabled)")


STREAMLIT PRODUCTION CONFIGURATION
[server]
headless = true
enableCORS = true
enableXsrfProtection = true
maxUploadSize = 50

[theme]
primaryColor = "#1E3A8A"
backgroundColor = "#FFFFFF"
secondaryBackgroundColor = "#F3F4F6"
textColor = "#1F2937"
font = "sans serif"

[browser]
gatherUsageStats = false
Configuration Check:   PASS (Headless mode and secure CORS enabled)


---
### 7. Headless Streamlit Smoke Test Execution (`AppTest` API)

We simulate a complete application launch using Streamlit's official `AppTest` framework to verify that components, forms, and tabs initialize cleanly without uncaught exceptions.

In [6]:
from streamlit.testing.v1 import AppTest

app_path = ROOT_DIR / "app.py"
print(f"Launching headless AppTest for {app_path.name}...")
at = AppTest.from_file(str(app_path), default_timeout=25).run()

print(f"App Rendered Successfully: {not at.exception}")
print(f"Sidebar Elements Count:    {len(at.sidebar)}")
print(f"Tabs Rendered:             {len(at.tabs)}")
print(f"Form Submit Buttons:       {len(at.button)}")
assert not at.exception, f"AppTest failed with exception: {at.exception}"
print("\nLocal Launch Smoke Test:  PASS (Zero runtime exceptions)")


2026-11-02 14:02:59.262 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


Launching headless AppTest for app.py...


2026-11-02 14:02:59.453 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


2026-11-02 14:02:59.457 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


2026-11-02 14:02:59.459 Please replace `use_container_width` with `width`.

`use_container_width` will be removed after 2025-12-31.

For `use_container_width=True`, use `width='stretch'`. For `use_container_width=False`, use `width='content'`.


App Rendered Successfully: True


Sidebar Elements Count:    23
Tabs Rendered:             4
Form Submit Buttons:       2

Local Launch Smoke Test:  PASS (Zero runtime exceptions)


---
### 8. Cloud Deployment Step-by-Step Procedure

Follow these exact steps to complete cloud publication on Streamlit Community Cloud:

```bash
# Step 1: Ensure GitHub remote is set and push all files
git remote add origin https://github.com/<your-username>/<your-repo-name>.git
git branch -M main
git add notebooks/Day_22_Streamlit_App_Deployment.ipynb docs/Day_22_Streamlit_Deployment_Guide.md app.py .streamlit/config.toml
git add artifacts/student_performance_pipeline.joblib artifacts/student_performance_metadata.json
git commit -m "Day 22: Deploy student performance prediction app"
git push -u origin main
```

#### In Streamlit Community Cloud ([share.streamlit.io](https://share.streamlit.io/)):
1. Sign in using your GitHub account.
2. Click **"Create app"** (or **"New app"**).
3. In the deployment modal:
   - **Repository:** `your-username/your-repo-name`
   - **Branch:** `main`
   - **Main file path:** `app.py`
   - **Custom URL:** Choose a subdomain (e.g. `student-performance-ai`).
4. Click **"Deploy!"**.
5. View build logs in the right-hand flyout. In ~1–2 minutes, your app is accessible worldwide!

---
### 9. Post-Deployment Verification Checklist

Upon accessing your live hosted URL (`https://<subdomain>.streamlit.app`), execute this verification checklist:

- [ ] **1. Cloud App Accessibility:** Web page loads with header `Student Performance Prediction & Monitoring`.
- [ ] **2. Model Deserialization:** Sidebar displays `Decision Tree` architecture, CV F1 ($0.5237$), and test accuracy ($53.85%$) without red error banners.
- [ ] **3. Single-Student Prediction:** Enter survey features (e.g. Study time = 2, Failures = 0, Absences = 4) and verify predicted performance tier and probability meters render.
- [ ] **4. Leakage Prevention:** Verify that final grade $G3$ and performance category are never requested.
- [ ] **5. Batch CSV Upload:** Upload `day_17_prediction_demo.csv` (features only) in Tab 2 and verify batch processing and CSV export.
- [ ] **6. Monitoring & Privacy:** Open Tab 3 to verify telemetry aggregations. Expand the Telemetry Audit Log to verify strict anonymity (no student names or IDs).
- [ ] **7. Subjective Feedback:** Submit feedback (👍 / 👎) on a prediction and confirm acknowledgment.

---
### 10. Troubleshooting, Maintenance & Rollback Protocol

| Cloud Error | Cause | Remediation |
| :--- | :--- | :--- |
| `FileNotFoundError: student_performance_pipeline.joblib` | Model artifact not pushed to GitHub | Commit artifact: `git add artifacts/` and push. |
| `ModuleNotFoundError` | Missing package in `requirements.txt` | Add package to `requirements.txt` and push. |
| `InconsistentVersionWarning` | Scikit-learn unpickle version mismatch | Ensure `scikit-learn>=1.4.0,<2.0.0` in `requirements.txt`. |

#### Continuous Delivery & Rollback:
- **Automatic Redeploy:** Streamlit Cloud automatically pulls and redeploys whenever you push to `main`.
- **Instant Rollback:** If a bug occurs, run `git revert HEAD` and push; Streamlit Cloud rolls back immediately.
- **Reboot Cache:** In Streamlit Cloud, click **Manage app** -> **...** -> **Reboot app** to clear container memory.

---
### 11. Day 22 Development Log & Final Status

#### Day 22 Accomplishments:
1. **Pre-Deployment Auditing:** Verified that all 18 automated tests pass locally, model artifact (20.6 KB) deserializes properly, and `app.py` passes syntax compilation.
2. **Hosting Architecture Finalization:** Selected Streamlit Community Cloud for frictionless, cost-free GitHub integration with automatic continuous delivery.
3. **Server Configuration Hardening:** Updated `.streamlit/config.toml` to enforce `headless = true` and `enableCORS = true` in compliance with latest Streamlit security standards.
4. **Deployment Documentation:** Authored `docs/Day_22_Streamlit_Deployment_Guide.md` with complete manual deployment instructions, post-deployment checklist, and troubleshooting matrix.
5. **Headless Verification:** Completed automated launch smoke test via Streamlit `AppTest` API with zero runtime exceptions.

**Deployment Status:** **DEPLOYMENT PREPARED & READY FOR MANUAL LAUNCH**
*Live URL pending user authentication on Streamlit Community Cloud.*